# 08. Quantile-Stratified Surrogate

원모형 예측값을 5분위로 나누고, 각 구간별 depth-1 surrogate를 따로 학습.
구간 내 feature 중요도 패턴이 더 균일해져서 Top-4/AdvTop-4가 향상되는지 검증.

- Tree depth=1 (no mono / +mono)
- EBM GAM (no mono / +mono)
- 5분위: Q1(하위 20%) ~ Q5(상위 20%)

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate, EBMSurrogate

DATA_DIR = '../.data'
N_QUANTILES = 5

In [2]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }

In [3]:
def get_bb(data_flag):
    bb_contrib = base_models[data_flag]['bb_contribs']['lgb']
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    order = np.argsort(np.abs(bb_contrib), axis=1)[:, ::-1]
    bb_rank = feature_names[order]
    bb_adv = []
    for i in range(len(bb_contrib)):
        pos_idx = np.where(bb_contrib[i] > 0)[0]
        if len(pos_idx) == 0:
            bb_adv.append(np.array([], dtype='<U1'))
        else:
            pos_order = pos_idx[np.argsort(bb_contrib[i][pos_idx])[::-1]]
            bb_adv.append(feature_names[pos_order])
    return bb_contrib, bb_rank, bb_adv


def advfull_metrics(bb_adv_list, surr_adv_list):
    recalls, jaccards = [], []
    for i in range(len(bb_adv_list)):
        bb_set = set(bb_adv_list[i])
        su_set = set(surr_adv_list[i])
        if len(bb_set) == 0:
            continue
        inter = len(bb_set & su_set)
        recalls.append(inter / len(bb_set))
        union = len(bb_set | su_set)
        jaccards.append(inter / union if union > 0 else 0.0)
    r = round(float(np.mean(recalls)), 4) if recalls else 0.0
    j = round(float(np.mean(jaccards)), 4) if jaccards else 0.0
    return r, j


def evaluate_from_arrays(pred, contribs, y, bb_rank, bb_adv, feature_names, k=4):
    """Evaluate from pre-computed pred and contribs arrays."""
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    # Top-k
    order = np.argsort(np.abs(contribs), axis=1)[:, ::-1]
    surr_rank = feature_names[order]
    result[f'Top{k}'] = round(float(np.mean([
        len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(y))])), 4)
    # AdvTop-k
    surr_adv_list = []
    adv_ov = []
    for i in range(len(y)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(neg_idx) == 0:
            surr_adv_list.append(np.array([], dtype='<U1'))
        else:
            neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
            surr_adv_list.append(feature_names[neg_order])
        if len(bb_adv[i]) == 0 or len(surr_adv_list[-1]) == 0:
            continue
        ke = min(k, len(bb_adv[i]), len(surr_adv_list[-1]))
        adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv_list[-1][:ke]))/ke)
    result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0
    recall, jaccard = advfull_metrics(bb_adv, surr_adv_list)
    result['AdvFull_R'] = recall
    result['AdvFull_J'] = jaccard
    return result


def evaluate_surrogate(surr, X, y, bb_rank, bb_adv, k=4):
    """Evaluate surrogate object (03 compatible)."""
    pred = surr.predict(X)
    contribs = surr.contributions(X)
    feature_names = np.array(list(X.columns)) if hasattr(X, 'columns') else np.arange(X.shape[1]).astype(str)
    return evaluate_from_arrays(pred, contribs, y, bb_rank, bb_adv, feature_names, k)

In [4]:
%%time
for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val, y_test = t['y_logit_tr'], t['y_logit_val'], t['y_logit_test']
    bb_contrib, bb_rank, bb_adv = get_bb(data_flag)
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    n_features = len(feature_names)

    X_train = pd.concat([X_tr, X_val], ignore_index=True)
    y_train = np.concatenate([y_tr, y_val])

    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')

    surr_configs = [
        ('Tree-d1', False),
        ('Tree-d1+mono', True),
        ('EBM-GAM', False),
        ('EBM-GAM+mono', True),
    ]

    for surr_name, use_mono in surr_configs:
        is_tree = 'Tree' in surr_name
        mono_mode = 'auto' if use_mono else 'none'

        print(f'\n  === {surr_name} ===')

        # ── Global baseline (단일 모형) ──
        if is_tree:
            surr_global = TreeSurrogate(max_depth=1, monotone_detect_mode=mono_mode, verbose=0)
            surr_global.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        else:
            surr_global = EBMSurrogate(interactions=0, monotone_detect_mode=mono_mode)
            surr_global.fit(X_train, y_train)

        res_global = evaluate_surrogate(surr_global, X_test, y_test, bb_rank, bb_adv)
        print(f'    Global:     {res_global}')

        # ── Quantile-Stratified (5분위) ──
        # 분위 기준: training y_logit
        if is_tree:
            y_for_q = y_tr
        else:
            y_for_q = y_train
        quantile_edges = np.percentile(y_for_q, np.linspace(0, 100, N_QUANTILES + 1))

        # Test set 분위 배정
        q_assign_test = np.digitize(y_test, quantile_edges[1:-1])  # 0 ~ N_QUANTILES-1

        # 각 분위별 surrogate 학습
        pred_all = np.zeros(len(y_test))
        contribs_all = np.zeros((len(y_test), n_features))

        for q in range(N_QUANTILES):
            # Train 분위 마스크
            if is_tree:
                q_mask_tr = np.digitize(y_tr, quantile_edges[1:-1]) == q
                q_mask_val = np.digitize(y_val, quantile_edges[1:-1]) == q
                X_tr_q = X_tr[q_mask_tr]
                y_tr_q = y_tr[q_mask_tr]
                X_val_q = X_val[q_mask_val]
                y_val_q = y_val[q_mask_val]
                if len(X_tr_q) < 50 or len(X_val_q) < 10:
                    continue
                surr_q = TreeSurrogate(max_depth=1, monotone_detect_mode=mono_mode, verbose=0)
                surr_q.fit(X_tr_q, y_tr_q, eval_set=(X_val_q, y_val_q))
            else:
                q_mask_train = np.digitize(y_train, quantile_edges[1:-1]) == q
                X_train_q = X_train[q_mask_train]
                y_train_q = y_train[q_mask_train]
                if len(X_train_q) < 50:
                    continue
                surr_q = EBMSurrogate(interactions=0, monotone_detect_mode=mono_mode)
                surr_q.fit(X_train_q, y_train_q)

            # Test 분위 예측
            q_mask_test = q_assign_test == q
            if q_mask_test.sum() == 0:
                continue
            X_test_q = X_test[q_mask_test] if not hasattr(X_test, 'iloc') else X_test.iloc[q_mask_test.nonzero()[0]]
            pred_all[q_mask_test] = surr_q.predict(X_test_q)
            contribs_all[q_mask_test] = surr_q.contributions(X_test_q)

        res_quantile = evaluate_from_arrays(
            pred_all, contribs_all, y_test, bb_rank, bb_adv, feature_names)
        print(f'    Quantile-5: {res_quantile}')

        # 분위별 상세
        print(f'    --- per quantile ---')
        for q in range(N_QUANTILES):
            q_mask = q_assign_test == q
            n_q = q_mask.sum()
            if n_q < 10:
                continue
            res_q = evaluate_from_arrays(
                pred_all[q_mask], contribs_all[q_mask],
                y_test[q_mask], bb_rank[q_mask],
                [bb_adv[i] for i in range(len(bb_adv)) if q_mask[i]],
                feature_names)
            q_lo = quantile_edges[q]
            q_hi = quantile_edges[q+1]
            print(f'      Q{q+1} [{q_lo:.2f},{q_hi:.2f}) n={n_q}: {res_q}')


################################################################################
  GMSC
################################################################################

  === Tree-d1 ===


    Global:     {'R2': 0.9403, 'Spearman': 0.9738, 'Top4': 0.8638, 'AdvTop4': 0.9069, 'AdvFull_R': 0.9044, 'AdvFull_J': 0.8379}


    Quantile-5: {'R2': 0.9784, 'Spearman': 0.9908, 'Top4': 0.6672, 'AdvTop4': 0.8063, 'AdvFull_R': 0.8399, 'AdvFull_J': 0.7286}
    --- per quantile ---
      Q1 [-1.82,2.41) n=6142: {'R2': 0.8712, 'Spearman': 0.9112, 'Top4': 0.6333, 'AdvTop4': 0.8355, 'AdvFull_R': 0.7486, 'AdvFull_J': 0.732}
      Q2 [2.41,3.34) n=5942: {'R2': 0.4204, 'Spearman': 0.6931, 'Top4': 0.7359, 'AdvTop4': 0.8489, 'AdvFull_R': 0.845, 'AdvFull_J': 0.8059}
      Q3 [3.34,4.11) n=5901: {'R2': 0.4754, 'Spearman': 0.7023, 'Top4': 0.6842, 'AdvTop4': 0.916, 'AdvFull_R': 0.8907, 'AdvFull_J': 0.8452}


      Q4 [4.11,4.59) n=5962: {'R2': 0.3463, 'Spearman': 0.6283, 'Top4': 0.624, 'AdvTop4': 0.8093, 'AdvFull_R': 0.8989, 'AdvFull_J': 0.7127}
      Q5 [4.59,5.34) n=6053: {'R2': 0.7922, 'Spearman': 0.8979, 'Top4': 0.6601, 'AdvTop4': 0.5896, 'AdvFull_R': 0.8162, 'AdvFull_J': 0.5169}

  === Tree-d1+mono ===


    Global:     {'R2': 0.9332, 'Spearman': 0.969, 'Top4': 0.8467, 'AdvTop4': 0.8988, 'AdvFull_R': 0.8366, 'AdvFull_J': 0.7583}


...

      Q1 [-1.06,2.01) n=12282: {'R2': 0.6714, 'Spearman': 0.7732, 'Top4': 0.6449, 'AdvTop4': 0.6836, 'AdvFull_R': 0.75, 'AdvFull_J': 0.5848}


      Q2 [2.01,2.62) n=12484: {'R2': 0.2068, 'Spearman': 0.4697, 'Top4': 0.6476, 'AdvTop4': 0.6603, 'AdvFull_R': 0.7628, 'AdvFull_J': 0.5997}


      Q3 [2.62,3.08) n=12510: {'R2': 0.1641, 'Spearman': 0.4193, 'Top4': 0.677, 'AdvTop4': 0.6935, 'AdvFull_R': 0.8081, 'AdvFull_J': 0.6484}


      Q4 [3.08,3.56) n=12201: {'R2': 0.1592, 'Spearman': 0.412, 'Top4': 0.4943, 'AdvTop4': 0.5355, 'AdvFull_R': 0.8077, 'AdvFull_J': 0.563}


      Q5 [3.56,5.77) n=12026: {'R2': 0.4942, 'Spearman': 0.6663, 'Top4': 0.4844, 'AdvTop4': 0.4713, 'AdvFull_R': 0.7462, 'AdvFull_J': 0.4391}
CPU times: total: 22min 31s
Wall time: 18min
